# 🏦 Clustering Clients Fintech

> **Objectif** : Segmenter automatiquement les clients d'une institution financière
> à partir de leurs comportements transactionnels, afin d'obtenir des profils
> actionnables pour le marketing, la gestion des risques et la fidélisation.

---

## Plan du notebook

| # | Section | Description |
|---|---------|-------------|
| 1 | ⚙️ Configuration | Installation, imports, chemins de fichiers |
| 2 | 📥 Chargement des données | Lecture + validation des CSV |
| 3 | 🔍 Analyse exploratoire (EDA) | Distributions, valeurs manquantes, corrélations |
| 4 | 🔧 Feature Engineering | Calcul des indicateurs RFM + comportementaux |
| 5 | 🧹 Préprocessing | Nettoyage, encodage, normalisation |
| 6 | 📐 Réduction dimensionnelle | PCA pour visualisation et clustering |
| 7 | 🤖 Clustering | K-Means, HDBSCAN, Gaussian Mixture |
| 8 | 📊 Évaluation | Silhouette, courbe du coude, Davies-Bouldin |
| 9 | 👁️ Visualisations | Scatter plots, heatmaps des profils de clusters |
| 10 | 💼 Interprétation métier | Personas et recommandations par segment |
| 11 | 💾 Export | Sauvegarde des résultats |


---
## 1 ⚙️ Configuration

### 1.1 Installation des dépendances

La cellule ci-dessous installe les bibliothèques nécessaires.
Sur **Google Colab** elles s'installent automatiquement ; en local, préférez :
```bash
pip install -r requirements.txt
```


In [ ]:
# ── Installation (Colab / environnement vierge) ───────────────────────────────
import subprocess, sys

REQUIRED = [
    "hdbscan>=0.8.33",
    "umap-learn>=0.5.4",
    "plotly>=5.18",
]

for pkg in REQUIRED:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

print("✔ Dépendances prêtes.")


### 1.2 Imports


In [ ]:
# ── Bibliothèques standard ────────────────────────────────────────────────────
import warnings
import os
from pathlib import Path

warnings.filterwarnings("ignore")

# ── Data ──────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})
sns.set_theme(style="whitegrid", palette="muted")

# ── Machine learning ──────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.pipeline import Pipeline
import hdbscan

# ── Réproductibilité ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

print("✔ Imports réussis.")


### 1.3 Chemins de fichiers

**Modifiez les chemins ci-dessous pour pointer vers vos propres données.**
Si les fichiers n'existent pas encore, le script de génération sera lancé automatiquement.


In [ ]:
# ── Chemins (à adapter selon votre environnement) ────────────────────────────

# Option 1 : dépôt cloné localement ou sur Colab avec git clone
REPO_ROOT = Path(".")  # racine du dépôt

# Option 2 : Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# REPO_ROOT = Path("/content/drive/MyDrive/D-fi-de-la-nuit-2024")

CUSTOMERS_PATH    = REPO_ROOT / "data" / "customers.csv"
TRANSACTIONS_PATH = REPO_ROOT / "data" / "transactions.csv"
RESULTS_PATH      = REPO_ROOT / "data" / "customers_segmented.csv"

# ── Génération automatique des données synthétiques si nécessaire ─────────────
if not CUSTOMERS_PATH.exists() or not TRANSACTIONS_PATH.exists():
    print("⚠️  Fichiers de données introuvables. Génération des données synthétiques…")
    gen_script = REPO_ROOT / "data" / "generate_sample_data.py"
    if gen_script.exists():
        import subprocess, sys
        subprocess.run([sys.executable, str(gen_script)], check=True)
    else:
        print("❌  Script de génération introuvable. Veuillez fournir vos propres données.")
else:
    print(f"✔ Données trouvées :")
    print(f"   {CUSTOMERS_PATH}")
    print(f"   {TRANSACTIONS_PATH}")


---
## 2 📥 Chargement et validation des données


In [ ]:
# ── Chargement ────────────────────────────────────────────────────────────────
customers    = pd.read_csv(CUSTOMERS_PATH, parse_dates=["registration_date"])
transactions = pd.read_csv(TRANSACTIONS_PATH, parse_dates=["date"])

print(f"Clients      : {customers.shape[0]:,} lignes × {customers.shape[1]} colonnes")
print(f"Transactions : {transactions.shape[0]:,} lignes × {transactions.shape[1]} colonnes")


In [ ]:
# ── Aperçu des clients ────────────────────────────────────────────────────────
customers.head()


In [ ]:
# ── Aperçu des transactions ───────────────────────────────────────────────────
transactions.head()


In [ ]:
# ── Validation basique ────────────────────────────────────────────────────────
def validate_dataframe(df: pd.DataFrame, name: str) -> None:
    missing_pct = df.isnull().mean() * 100
    print(f"\n{'='*50}")
    print(f"  {name}  — {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
    print(f"{'='*50}")
    print(df.dtypes.to_string())
    print("\nValeurs manquantes (%) :")
    print(missing_pct[missing_pct > 0].to_string() if missing_pct.any() else "  Aucune ✔")
    print("\nDoublons :", df.duplicated().sum())

validate_dataframe(customers, "customers")
validate_dataframe(transactions, "transactions")


---
## 3 🔍 Analyse Exploratoire (EDA)


In [ ]:
# ── Distribution des montants (log-scale car très skewée) ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(transactions["amount"], bins=80, color="steelblue", edgecolor="none", alpha=0.8)
axes[0].set_title("Distribution des montants (échelle linéaire)")
axes[0].set_xlabel("Montant")

axes[1].hist(np.log1p(transactions["amount"]), bins=80, color="darkorange", edgecolor="none", alpha=0.8)
axes[1].set_title("Distribution des montants (log1p)")
axes[1].set_xlabel("log(1 + Montant)")

plt.tight_layout()
plt.show()


In [ ]:
# ── Transactions par canal et par type ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

transactions["channel"].value_counts().plot(kind="bar", ax=axes[0], color="steelblue", rot=0)
axes[0].set_title("Transactions par canal")
axes[0].set_ylabel("Nombre")

transactions["type"].value_counts().plot(kind="bar", ax=axes[1], color="darkorange", rot=15)
axes[1].set_title("Transactions par type")
axes[1].set_ylabel("Nombre")

plt.tight_layout()
plt.show()


In [ ]:
# ── Évolution temporelle du volume de transactions ───────────────────────────
monthly = (
    transactions
    .assign(month=transactions["date"].dt.to_period("M"))
    .groupby("month")
    .size()
    .reset_index(name="n_transactions")
)
monthly["month"] = monthly["month"].astype(str)

fig = px.line(monthly, x="month", y="n_transactions",
              title="Volume mensuel de transactions",
              labels={"month": "Mois", "n_transactions": "Nombre de transactions"},
              markers=True)
fig.show()


In [ ]:
# ── Distribution des soldes clients ──────────────────────────────────────────
if "avg_balance" in customers.columns:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    customers["avg_balance"].plot(kind="hist", bins=60, ax=axes[0], color="teal", alpha=0.8)
    axes[0].set_title("Distribution du solde moyen")
    axes[0].set_xlabel("Solde (€)")

    np.log1p(customers["avg_balance"]).plot(kind="hist", bins=60, ax=axes[1],
                                            color="mediumorchid", alpha=0.8)
    axes[1].set_title("Distribution du solde moyen (log1p)")
    axes[1].set_xlabel("log(1 + Solde)")
    plt.tight_layout()
    plt.show()


---
## 4 🔧 Feature Engineering

On calcule des **indicateurs RFM** (Recency, Frequency, Monetary) enrichis
de métriques comportementales spécifiques au contexte fintech.

| Feature | Description |
|---------|-------------|
| `recency_days` | Nb de jours depuis la dernière transaction |
| `frequency` | Nb total de transactions sur la période |
| `monetary_total` | Montant total des transactions |
| `monetary_avg` | Montant moyen par transaction |
| `monetary_std` | Écart-type des montants (volatilité) |
| `monetary_max` | Montant maximum (détection de pics) |
| `pct_international` | % de transactions internationales |
| `pct_mobile` | % de transactions via mobile |
| `pct_flagged` | % de transactions signalées |
| `n_types_used` | Diversité des types de transactions |
| `tenure_days` | Ancienneté du client (si disponible) |
| `avg_balance` | Solde moyen (si disponible) |


In [ ]:
# ── Date de référence : dernier jour de la période ───────────────────────────
REFERENCE_DATE = transactions["date"].max() + pd.Timedelta(days=1)

# ── Agrégations de base ───────────────────────────────────────────────────────
rfm = (
    transactions
    .groupby("customer_id")
    .agg(
        recency_days    = ("date", lambda x: (REFERENCE_DATE - x.max()).days),
        frequency       = ("transaction_id", "count"),
        monetary_total  = ("amount", "sum"),
        monetary_avg    = ("amount", "mean"),
        monetary_std    = ("amount", "std"),
        monetary_max    = ("amount", "max"),
        pct_international = ("is_international", "mean") if "is_international" in transactions.columns else ("amount", lambda x: 0),
        pct_flagged       = ("is_flagged",       "mean") if "is_flagged"       in transactions.columns else ("amount", lambda x: 0),
    )
    .reset_index()
)

# ── % transactions mobiles ────────────────────────────────────────────────────
if "channel" in transactions.columns:
    mobile_pct = (
        transactions
        .assign(is_mobile=transactions["channel"].eq("mobile"))
        .groupby("customer_id")["is_mobile"]
        .mean()
        .rename("pct_mobile")
    )
    rfm = rfm.merge(mobile_pct, on="customer_id", how="left")
else:
    rfm["pct_mobile"] = 0.0

# ── Diversité des types de transactions ──────────────────────────────────────
if "type" in transactions.columns:
    type_diversity = (
        transactions
        .groupby("customer_id")["type"]
        .nunique()
        .rename("n_types_used")
    )
    rfm = rfm.merge(type_diversity, on="customer_id", how="left")
else:
    rfm["n_types_used"] = 1

# ── Fusion avec les attributs clients ────────────────────────────────────────
extra_cols = ["customer_id"]
if "tenure_days"  in customers.columns: extra_cols.append("tenure_days")
if "avg_balance"  in customers.columns: extra_cols.append("avg_balance")
if "country"      in customers.columns: extra_cols.append("country")
if "account_type" in customers.columns: extra_cols.append("account_type")

rfm = rfm.merge(customers[extra_cols], on="customer_id", how="left")

# ── Imputation des NaN (clients sans transactions multiples) ──────────────────
rfm["monetary_std"] = rfm["monetary_std"].fillna(0)

print(f"Feature matrix : {rfm.shape[0]:,} clients × {rfm.shape[1]} colonnes")
rfm.describe().T


---
## 5 🧹 Préprocessing

### Étapes :
1. **Séparation** features numériques / catégorielles
2. **Log-transformation** des variables très skewées (montants)
3. **Encodage One-Hot** des variables catégorielles
4. **RobustScaler** (résistant aux outliers) sur toutes les variables numériques


In [ ]:
# ── Colonnes à utiliser pour le clustering ───────────────────────────────────
NUMERIC_FEATURES = [
    "recency_days", "frequency",
    "monetary_total", "monetary_avg", "monetary_std", "monetary_max",
    "pct_international", "pct_mobile", "pct_flagged", "n_types_used",
]
if "tenure_days" in rfm.columns: NUMERIC_FEATURES.append("tenure_days")
if "avg_balance" in rfm.columns: NUMERIC_FEATURES.append("avg_balance")

CATEGORICAL_FEATURES = []
if "account_type" in rfm.columns: CATEGORICAL_FEATURES.append("account_type")
# Note : "country" est souvent trop granulaire ; on peut l'inclure si peu de modalités.

# ── Log-transformation des variables skewées ─────────────────────────────────
LOG_COLS = ["monetary_total", "monetary_avg", "monetary_std", "monetary_max"]
if "avg_balance" in NUMERIC_FEATURES:
    LOG_COLS.append("avg_balance")

df_proc = rfm[["customer_id"] + NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()

for col in LOG_COLS:
    if col in df_proc.columns:
        df_proc[col] = np.log1p(df_proc[col].clip(lower=0))

# ── Encodage One-Hot ──────────────────────────────────────────────────────────
if CATEGORICAL_FEATURES:
    df_proc = pd.get_dummies(df_proc, columns=CATEGORICAL_FEATURES, drop_first=False)

feature_cols = [c for c in df_proc.columns if c != "customer_id"]
X_raw = df_proc[feature_cols].values

# ── Standardisation (RobustScaler) ────────────────────────────────────────────
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f"Matrice finale : {X_scaled.shape[0]:,} lignes × {X_scaled.shape[1]} features")
print("Features :", feature_cols)


---
## 6 📐 Réduction dimensionnelle (PCA)

La PCA sert à deux fins :
- **Visualisation** en 2D pour interpréter les clusters
- **Débruitage** avant clustering (optionnel — conserve 95 % de la variance)


In [ ]:
# ── PCA pour le clustering (garde 95 % de la variance) ───────────────────────
pca_full = PCA(n_components=0.95, random_state=SEED)
X_pca    = pca_full.fit_transform(X_scaled)

cumvar = np.cumsum(pca_full.explained_variance_ratio_) * 100

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, len(cumvar) + 1), cumvar, marker="o", color="steelblue")
ax.axhline(95, color="red", linestyle="--", label="95 %")
ax.set_title("Variance expliquée cumulée — PCA")
ax.set_xlabel("Nombre de composantes")
ax.set_ylabel("Variance expliquée (%)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Dimensions retenues pour 95 % de variance : {X_pca.shape[1]}")


In [ ]:
# ── PCA 2D pour la visualisation ─────────────────────────────────────────────
pca_2d  = PCA(n_components=2, random_state=SEED)
X_2d    = pca_2d.fit_transform(X_scaled)

print(f"Variance expliquée par PC1 : {pca_2d.explained_variance_ratio_[0]:.1%}")
print(f"Variance expliquée par PC2 : {pca_2d.explained_variance_ratio_[1]:.1%}")


---
## 7 🤖 Clustering

On entraîne trois algorithmes complémentaires :

| Algorithme | Forces | Limites |
|---|---|---|
| **K-Means** | Rapide, interprétable, segments équilibrés | Suppose des clusters sphériques, sensible aux outliers |
| **HDBSCAN** | Détecte le bruit/outliers, formes quelconques | Moins interprétable, paramètre `min_cluster_size` |
| **Gaussian Mixture (GMM)** | Probabiliste (appartenance soft), flexible | Plus lent, peut sur-fitter |


### 7.1 Choix du nombre de clusters — K-Means (méthode du coude + silhouette)


In [ ]:
K_RANGE = range(2, 11)

inertias      = []
silhouettes   = []
davies_bouldin= []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_pca, labels, sample_size=min(2000, len(labels))))
    davies_bouldin.append(davies_bouldin_score(X_pca, labels))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(K_RANGE, inertias, marker="o", color="steelblue")
axes[0].set_title("Méthode du coude (Inertie)")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertie")

axes[1].plot(K_RANGE, silhouettes, marker="o", color="darkorange")
axes[1].set_title("Score Silhouette (↑ meilleur)")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette")

axes[2].plot(K_RANGE, davies_bouldin, marker="o", color="green")
axes[2].set_title("Davies-Bouldin (↓ meilleur)")
axes[2].set_xlabel("k")
axes[2].set_ylabel("DB Score")

plt.suptitle("Sélection du nombre de clusters K-Means", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

best_k = K_RANGE.start + int(np.argmax(silhouettes))
print(f"\n➡  Meilleur k selon silhouette : {best_k}")


### 7.2 K-Means final


In [ ]:
# ── Modifiez N_CLUSTERS selon l'analyse ci-dessus ─────────────────────────────
N_CLUSTERS = best_k  # ou forcer : N_CLUSTERS = 4

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=SEED, n_init=20, max_iter=500)
kmeans_labels = kmeans.fit_predict(X_pca)

rfm["cluster_kmeans"] = kmeans_labels
print(f"K-Means ({N_CLUSTERS} clusters) — distribution :")
print(rfm["cluster_kmeans"].value_counts().sort_index().to_string())


### 7.3 HDBSCAN


In [ ]:
# ── HDBSCAN ──────────────────────────────────────────────────────────────────
# min_cluster_size : taille minimale d'un cluster
# -1 = outliers (points non assignés à un cluster)
hdb = hdbscan.HDBSCAN(
    min_cluster_size=max(20, len(X_pca) // 50),
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
)
hdb_labels = hdb.fit_predict(X_pca)

rfm["cluster_hdbscan"] = hdb_labels

n_clusters_hdb = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
n_noise        = (hdb_labels == -1).sum()
print(f"HDBSCAN → {n_clusters_hdb} clusters, {n_noise} outliers ({n_noise/len(hdb_labels):.1%})")
print(rfm["cluster_hdbscan"].value_counts().sort_index().to_string())


### 7.4 Gaussian Mixture Model (GMM)


In [ ]:
# ── GMM ──────────────────────────────────────────────────────────────────────
gmm = GaussianMixture(
    n_components=N_CLUSTERS,
    covariance_type="full",
    random_state=SEED,
    max_iter=200,
    n_init=5,
)
gmm.fit(X_pca)
gmm_labels = gmm.predict(X_pca)
gmm_proba  = gmm.predict_proba(X_pca)  # probabilité d'appartenance à chaque cluster

rfm["cluster_gmm"] = gmm_labels
print(f"GMM ({N_CLUSTERS} composantes) — distribution :")
print(rfm["cluster_gmm"].value_counts().sort_index().to_string())


---
## 8 📊 Évaluation des clusters


In [ ]:
# ── Métriques comparatives ────────────────────────────────────────────────────
results = []

for name, labels in [
    ("K-Means",  kmeans_labels),
    ("HDBSCAN",  hdb_labels),
    ("GMM",      gmm_labels),
]:
    mask = labels != -1  # exclure les outliers HDBSCAN
    if mask.sum() < 2 or len(set(labels[mask])) < 2:
        results.append({"Modèle": name, "Silhouette": None, "Davies-Bouldin": None, "Calinski-H.": None})
        continue
    sil = silhouette_score(X_pca[mask], labels[mask], sample_size=min(2000, mask.sum()))
    db  = davies_bouldin_score(X_pca[mask], labels[mask])
    ch  = calinski_harabasz_score(X_pca[mask], labels[mask])
    results.append({"Modèle": name, "Silhouette (↑)": round(sil, 3),
                    "Davies-Bouldin (↓)": round(db, 3),
                    "Calinski-Harabasz (↑)": round(ch, 0)})

eval_df = pd.DataFrame(results).set_index("Modèle")
print(eval_df.to_string())
eval_df


---
## 9 👁️ Visualisations des clusters


In [ ]:
def plot_clusters_2d(X2d, labels, title, ax=None):
    """Scatter plot 2D coloré par cluster."""
    show = ax is None
    if show:
        fig, ax = plt.subplots(figsize=(8, 6))
    unique_labels = sorted(set(labels))
    cmap = cm.get_cmap("tab10", len(unique_labels))
    for i, lbl in enumerate(unique_labels):
        mask = labels == lbl
        color = "grey" if lbl == -1 else cmap(i)
        label_str = "Outliers" if lbl == -1 else f"Cluster {lbl}"
        ax.scatter(X2d[mask, 0], X2d[mask, 1], c=[color], s=8, alpha=0.5, label=label_str)
    ax.set_title(title)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.legend(markerscale=3, fontsize=8)
    if show:
        plt.tight_layout()
        plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_clusters_2d(X_2d, kmeans_labels, f"K-Means  (k={N_CLUSTERS})", ax=axes[0])
plot_clusters_2d(X_2d, hdb_labels,    "HDBSCAN",                    ax=axes[1])
plot_clusters_2d(X_2d, gmm_labels,    f"GMM  (k={N_CLUSTERS})",     ax=axes[2])
plt.suptitle("Projection PCA 2D — comparaison des algorithmes", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


### 9.1 Profils des clusters K-Means — Heatmap


In [ ]:
# ── Profils moyens par cluster (features originales, non scalées) ─────────────
PROFILE_FEATURES = [
    "recency_days", "frequency",
    "monetary_avg", "monetary_max",
    "pct_mobile", "pct_international",
]
if "tenure_days" in rfm.columns: PROFILE_FEATURES.append("tenure_days")
if "avg_balance" in rfm.columns: PROFILE_FEATURES.append("avg_balance")

cluster_profiles = (
    rfm.groupby("cluster_kmeans")[PROFILE_FEATURES]
    .mean()
    .round(2)
)

# Normalisation pour la heatmap (Z-score par feature)
profiles_norm = (cluster_profiles - cluster_profiles.mean()) / cluster_profiles.std()

fig, ax = plt.subplots(figsize=(12, max(4, N_CLUSTERS * 0.8)))
sns.heatmap(
    profiles_norm.T, annot=cluster_profiles.T.round(1),
    fmt="g", cmap="RdYlGn_r", center=0, ax=ax,
    linewidths=0.5, cbar_kws={"label": "Z-score"}
)
ax.set_title("Profils moyens par cluster K-Means\n(couleur = Z-score, valeur = moyenne réelle)")
ax.set_xlabel("Cluster")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()


### 9.2 Boxplots des features clés


In [ ]:
key_features = ["recency_days", "frequency", "monetary_avg"]
fig, axes = plt.subplots(1, len(key_features), figsize=(14, 5))

for i, feat in enumerate(key_features):
    if feat in rfm.columns:
        rfm.boxplot(column=feat, by="cluster_kmeans", ax=axes[i],
                    notch=False, patch_artist=True,
                    medianprops={"color": "black"})
        axes[i].set_title(feat)
        axes[i].set_xlabel("Cluster K-Means")

plt.suptitle("Distribution des features par cluster", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()


### 9.3 Visualisation interactive Plotly


In [ ]:
plot_df = pd.DataFrame({
    "PC1": X_2d[:, 0],
    "PC2": X_2d[:, 1],
    "Cluster": kmeans_labels.astype(str),
    "frequency": rfm["frequency"].values,
    "monetary_avg": rfm["monetary_avg"].values.round(2),
    "customer_id": rfm["customer_id"].values,
})

fig = px.scatter(
    plot_df, x="PC1", y="PC2", color="Cluster",
    hover_data=["customer_id", "frequency", "monetary_avg"],
    title=f"K-Means ({N_CLUSTERS} clusters) — Projection PCA 2D",
    width=900, height=550,
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_traces(marker=dict(size=5, opacity=0.6))
fig.show()


---
## 10 💼 Interprétation métier des clusters

La cellule ci-dessous génère automatiquement un tableau de profils
et propose des noms de segments selon les caractéristiques observées.
**Adaptez les labels à vos propres données !**


In [ ]:
# ── Tableau de profils complet ────────────────────────────────────────────────
all_features = [c for c in rfm.columns if c not in
                ["customer_id", "cluster_kmeans", "cluster_hdbscan", "cluster_gmm",
                 "segment_true", "country", "account_type"]]

full_profile = (
    rfm.groupby("cluster_kmeans")[all_features]
    .agg(["mean", "median"])
    .round(2)
)
print("Profil complet par cluster K-Means :")
full_profile


In [ ]:
# ── Nommage automatique heuristique des segments ──────────────────────────────
# Logique simple ; à affiner selon vos observations.

def name_segment(row):
    rec   = row["recency_days"]
    freq  = row["frequency"]
    money = row["monetary_avg"]
    pct_f = row.get("pct_flagged", 0)
    bal   = row.get("avg_balance", 0)

    if pct_f > 0.03:
        return "🚨 Clients à risque"
    if rec < 30 and freq > 10 and money > 300:
        return "⭐ Clients Premium Actifs"
    if rec < 60 and freq >= 5:
        return "✅ Clients Réguliers"
    if rec > 180 or freq < 2:
        return "😴 Clients Inactifs / Dormants"
    if money < 50:
        return "💰 Micro-transactions"
    return "📊 Clients Moyens"

# Calcule le profil pour le nommage
profile_for_naming = rfm.groupby("cluster_kmeans")[
    [c for c in ["recency_days", "frequency", "monetary_avg", "pct_flagged", "avg_balance"]
     if c in rfm.columns]
].mean()

segment_names = {k: name_segment(row) for k, row in profile_for_naming.iterrows()}
rfm["segment_name"] = rfm["cluster_kmeans"].map(segment_names)

print("\nAssignation des segments :")
display_df = rfm.groupby(["cluster_kmeans", "segment_name"]).size().reset_index(name="n_clients")
display_df["% clients"] = (display_df["n_clients"] / len(rfm) * 100).round(1)
print(display_df.to_string(index=False))


In [ ]:
# ── Camembert de la répartition des segments ──────────────────────────────────
seg_counts = rfm["segment_name"].value_counts()

fig = px.pie(
    values=seg_counts.values,
    names=seg_counts.index,
    title="Répartition des clients par segment",
    color_discrete_sequence=px.colors.qualitative.Pastel,
)
fig.update_traces(textposition="inside", textinfo="percent+label")
fig.show()


### Recommandations par segment

| Segment | Recommandation |
|---|---|
| ⭐ Premium Actifs | Programme de fidélité, offres premium, cross-sell investissement |
| ✅ Clients Réguliers | Encourager l'augmentation de fréquence, upsell assurances |
| 😴 Inactifs / Dormants | Campagne de réactivation, offre de bienvenue retour |
| 💰 Micro-transactions | Offre de compte micro-épargne, éducation financière |
| 🚨 À risque | Monitoring renforcé, vérification KYC, contact proactif |


---
## 11 💾 Export des résultats


In [ ]:
# ── Sauvegarde du CSV enrichi ─────────────────────────────────────────────────
export_cols = ["customer_id", "recency_days", "frequency", "monetary_avg",
               "cluster_kmeans", "cluster_hdbscan", "cluster_gmm", "segment_name"]
export_cols = [c for c in export_cols if c in rfm.columns]

rfm[export_cols].to_csv(RESULTS_PATH, index=False)
print(f"✔ Résultats sauvegardés → {RESULTS_PATH}")
rfm[export_cols].head()


---
## 🏁 Pistes d'amélioration futures

- [ ] **UMAP** à la place de PCA pour une meilleure visualisation non-linéaire
- [ ] **Optimisation HDBSCAN** : grid-search sur `min_cluster_size`
- [ ] **Stabilité des clusters** : bootstrap / multi-run sur sous-échantillons
- [ ] **Features avancées** : séries temporelles (tendance 30j/90j), graph de réseau de paiements
- [ ] **MLflow / Neptune** : tracking des expériences de clustering
- [ ] **Pipeline scikit-learn** : encapsuler preprocessing + clustering pour la production
- [ ] **Dashboard** : Streamlit ou Dash pour explorer les segments en temps réel
- [ ] **Données réelles** : connecter à une base SQL, API ou fichier Parquet
